# Executive Summary – Jupyter Test Notebook

Mirrors `executive_summary_tab.py` logic for local testing.

**Workflow**
1. Run Cell 1 – install dependencies (if needed)
2. Run Cell 2 – enter GP password when prompted
3. Run Cell 3 – enter OpenAI API key when prompted
4. Run Cell 4 – select month/year when prompted
5. Run Cell 5 – query Greenplum for current + next month data
6. Run Cell 6 – inspect the data
7. Run Cell 7 – build the LLM prompt
8. Run Cell 8 – call OpenAI and display the summary
9. Run Cell 9 – optionally save to a .txt file

In [ ]:
# ── Cell 1: Install / verify dependencies ─────────────────────────────────
# Uncomment and run once if packages are missing in your environment.
# !pip install psycopg2-binary openai pandas

In [ ]:
# ── Cell 2: Imports & Greenplum connection config ─────────────────────────
import getpass
from datetime import datetime

import pandas as pd
import psycopg2
from openai import OpenAI

# ── Greenplum connection details ──────────────────────────────────────────
GREENPLUM_HOST   = "greenplum-rdsp.zur.swissbank.com"
GREENPLUM_PORT   = 5432
GREENPLUM_DB     = "gprdsp"
GREENPLUM_USER   = "ds_rdsp_dev"
GREENPLUM_SCHEMA = "core_ikg"

GREENPLUM_PASSWORD = getpass.getpass("Enter Greenplum password: ")

def get_gp_conn():
    """Return a fresh psycopg2 connection to Greenplum."""
    return psycopg2.connect(
        host=GREENPLUM_HOST,
        port=GREENPLUM_PORT,
        dbname=GREENPLUM_DB,
        user=GREENPLUM_USER,
        password=GREENPLUM_PASSWORD,
    )

print("GP config ready.")

In [ ]:
# ── Cell 3: OpenAI configuration ──────────────────────────────────────────
OPENAI_API_KEY  = getpass.getpass("Enter OpenAI / Azure API key: ")
OPENAI_BASE_URL = "https://cirruspl-staat-ste-dev-ai.openai.azure.com/openai/v1/"

MODEL_NAME  = "gpt-4.1"
MAX_TOKENS  = 12000
TEMPERATURE = 0.1

openai_client = OpenAI(api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)
print("OpenAI client ready.")

In [ ]:
# ── Cell 4: Month / year selection ────────────────────────────────────────
#
# First, display available months from GP so the user can choose.

STAAT_TABLE = f"{GREENPLUM_SCHEMA}.staat_insight_release"
ODM_TABLE   = f"{GREENPLUM_SCHEMA}.odm_release_details"

conn = get_gp_conn()
available_months_df = pd.read_sql_query(
    f"""
    SELECT DISTINCT TO_CHAR(prod_release_date::date, 'YYYY-MM') AS month_val
    FROM {STAAT_TABLE}
    WHERE prod_release_date IS NOT NULL
    ORDER BY month_val DESC
    """,
    conn,
)
conn.close()

print("Available months in STAAT (latest first):")
for i, val in enumerate(available_months_df["month_val"], start=1):
    try:
        label = pd.Period(val, "M").to_timestamp().strftime("%B %Y")
    except Exception:
        label = val
    print(f"  {i:>2}. {label}  ({val})")

print()
raw_input = input(
    "Enter the month to summarise (e.g. 'May 2026' or '2026-05'): "
).strip()

# Parse flexible input ─────────────────────────────────────────────────────
SELECTED_MONTH: str  # will hold 'YYYY-MM'
try:
    # Try 'YYYY-MM' directly
    p = pd.Period(raw_input, "M")
    SELECTED_MONTH = str(p)
except Exception:
    # Try natural formats like 'May 2026', 'May-2026', 'May/2026'
    cleaned = raw_input.replace("-", " ").replace("/", " ")
    dt = datetime.strptime(cleaned, "%B %Y")
    SELECTED_MONTH = dt.strftime("%Y-%m")

SELECTED_LABEL = pd.Period(SELECTED_MONTH, "M").to_timestamp().strftime("%B %Y")
print(f"\nSelected month: {SELECTED_LABEL}  [{SELECTED_MONTH}]")

In [ ]:
# ── Cell 5: Query Greenplum ────────────────────────────────────────────────
#
# Fetches the LATEST BATCH per iteration whose prod_release_date falls in
# the selected month, then LEFT JOINs ODM onto STAAT.

def build_month_query(month_value: str) -> str:
    """
    Return SQL that selects latest-batch rows from STAAT + ODM for a given
    prod_release_date month.

    Strategy
    --------
    1. Identify every iteration_end_date where STAAT has a prod_release_date
       in *month_value* (e.g. '2026-05').
    2. For each such iteration take MAX(batch) from both STAAT and ODM.
    3. LEFT JOIN ODM onto STAAT so issues with no ODM rows are still included.
    """
    return f"""
    WITH target_iterations AS (
        -- All iterations that touch the selected month
        SELECT
            iteration_end_date,
            MAX(batch) AS max_batch
        FROM {STAAT_TABLE}
        WHERE TO_CHAR(prod_release_date::date, 'YYYY-MM') = '{month_value}'
        GROUP BY iteration_end_date
    ),
    staat_latest AS (
        -- Latest batch rows from STAAT for those iterations
        SELECT s.*
        FROM {STAAT_TABLE} s
        INNER JOIN target_iterations ti
            ON  s.iteration_end_date = ti.iteration_end_date
            AND s.batch              = ti.max_batch
    ),
    odm_latest AS (
        -- Latest batch rows from ODM for those same iterations
        SELECT o.*
        FROM {ODM_TABLE} o
        INNER JOIN target_iterations ti
            ON  o.iteration_end_date = ti.iteration_end_date
            AND o.batch              = ti.max_batch
    )
    SELECT
        s.id_x,
        s.title,
        s.labels,
        s.issue_summary,
        s.state,
        s.weight,
        s.prod_release_date,
        s.iteration_end_date,
        s.iteration_start_date,
        s.batch,
        o.rule_name,
        o.target_type,
        o.change_type
    FROM staat_latest s
    LEFT JOIN odm_latest o
        ON  s.id_x               = o.issue_id
        AND s.iteration_end_date = o.iteration_end_date
    ORDER BY s.iteration_end_date, s.id_x
    """


# ── Fetch current month ────────────────────────────────────────────────────
conn = get_gp_conn()
df_current = pd.read_sql_query(build_month_query(SELECTED_MONTH), conn)
conn.close()

df_current["change_type"] = df_current["change_type"].replace({"added": "new"})
print(f"Current month rows : {len(df_current)}")

# ── Fetch next month ───────────────────────────────────────────────────────
NEXT_MONTH = str(pd.Period(SELECTED_MONTH, "M") + 1)
NEXT_LABEL = pd.Period(NEXT_MONTH, "M").to_timestamp().strftime("%B %Y")

conn = get_gp_conn()
df_next = pd.read_sql_query(build_month_query(NEXT_MONTH), conn)
conn.close()

df_next["change_type"] = df_next["change_type"].replace({"added": "new"})
print(f"Next month ({NEXT_LABEL}) rows : {len(df_next)}")

df_next_month = df_next if not df_next.empty else None

In [ ]:
# ── Cell 6: Inspect the data ───────────────────────────────────────────────

print(f"=== {SELECTED_LABEL} — shape: {df_current.shape} ===")
display(df_current.head(10))

print("\nUnique labels in current month:")
all_labels = (
    df_current["labels"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
    .value_counts()
)
print(all_labels.to_string())

print("\nRows where label = 'Top Feature':")
display(
    df_current[
        df_current["labels"].fillna("").str.contains(r"(?i)top feature", regex=True)
    ][["id_x", "title", "labels", "rule_name", "change_type"]]
)

print("\nRows where label = 'New Insight':")
display(
    df_current[
        df_current["labels"].fillna("").str.contains(r"(?i)new insight", regex=True)
    ][["id_x", "title", "labels", "rule_name", "change_type"]]
)

In [ ]:
# ── Cell 7: Build the LLM prompt ──────────────────────────────────────────

def _has_label(labels_value, target: str) -> bool:
    """Return True if *target* appears as one of the comma-separated labels."""
    if not labels_value or not isinstance(labels_value, str):
        return False
    parts = [lbl.strip().lower() for lbl in labels_value.split(",")]
    return target.lower() in parts


def _filter_by_label(df: pd.DataFrame, label: str) -> pd.DataFrame:
    """Subset *df* to rows whose labels column contains *label* exactly."""
    if df is None or df.empty or "labels" not in df.columns:
        return pd.DataFrame()
    mask = df["labels"].apply(lambda v: _has_label(v, label))
    return df.loc[mask]


def build_prompt(
    issues_df: pd.DataFrame,
    next_month_df: pd.DataFrame | None = None,
) -> str:
    """
    Build the LLM prompt.

    Label conventions
    -----------------
    * 'Top Feature'  → Section 2 (Top Feature of the Month)
    * 'New Insight'  → Section 3 (New Insights ranked by impact)
    """

    # ── Section 1: All stories ─────────────────────────────────────────────
    issues_lines: list[str] = []
    seen_ids: set = set()

    for _, row in issues_df.iterrows():
        id_x = row.get("id_x", "")
        if id_x and id_x not in seen_ids:
            seen_ids.add(id_x)
            issues_lines.append(
                f"Story #{len(issues_lines) + 1}:\n"
                f"Title:   {row.get('title', 'N/A')}\n"
                f"Summary: {row.get('issue_summary', 'N/A')}\n"
                f"State:   {row.get('state', 'N/A')}\n"
                f"Labels:  {row.get('labels', 'N/A')}\n"
                f"Weight:  {row.get('weight', 'N/A')}"
            )

    issues_text = (
        "\n\n".join(issues_lines[:20]) if issues_lines else "No issues data available."
    )

    # ── Section 2: Top Feature ─────────────────────────────────────────────
    top_feature_df = _filter_by_label(issues_df, "Top Feature")
    top_feature_lines: list[str] = []
    seen_top: set = set()

    for _, row in top_feature_df.iterrows():
        id_x = row.get("id_x", "")
        if id_x and id_x not in seen_top:
            seen_top.add(id_x)
            top_feature_lines.append(
                f"Top Feature #{len(top_feature_lines) + 1}:\n"
                f"Title:   {row.get('title', 'N/A')}\n"
                f"Summary: {row.get('issue_summary', 'N/A')}\n"
                f"Labels:  {row.get('labels', 'N/A')}\n"
                f"Rule:    {row.get('rule_name', 'N/A')}\n"
                f"Change:  {row.get('change_type', 'N/A')}"
            )

    top_feature_text = (
        "\n\n".join(top_feature_lines[:5])
        if top_feature_lines
        else "No issue labelled 'Top Feature' found for this period."
    )

    # ── Section 3: New Insights ────────────────────────────────────────────
    new_insight_df = _filter_by_label(issues_df, "New Insight")
    new_insights_lines: list[str] = []
    seen_rules: set = set()

    for _, row in new_insight_df.iterrows():
        rule = row.get("rule_name", "")
        if rule and rule not in seen_rules:
            seen_rules.add(rule)
            new_insights_lines.append(
                f"New Insight #{len(new_insights_lines) + 1}:\n"
                f"Rule Name:   {rule}\n"
                f"Target Type: {row.get('target_type', 'N/A')}\n"
                f"Change Type: {row.get('change_type', 'N/A')}\n"
                f"Title:       {row.get('title', 'N/A')}\n"
                f"Summary:     {row.get('issue_summary', 'N/A')}"
            )

    insights_text = (
        "\n\n".join(new_insights_lines[:15])
        if new_insights_lines
        else "No issues labelled 'New Insight' found for this period."
    )

    # ── Section 6: Looking Ahead ───────────────────────────────────────────
    next_month_lines: list[str] = []
    if next_month_df is not None and not next_month_df.empty:
        next_seen_rules: set = set()
        for _, row in next_month_df.iterrows():
            rule   = row.get("rule_name", "")
            change = row.get("change_type", "")
            if rule and rule not in next_seen_rules:
                next_seen_rules.add(rule)
                next_month_lines.append(
                    f"- Rule Name:   {rule}\n"
                    f"  Target Type: {row.get('target_type', 'N/A')}\n"
                    f"  Change Type: {change}\n"
                    f"  Title:       {row.get('title', 'N/A')}\n"
                    f"  Labels:      {row.get('labels', 'N/A')}"
                )

    next_month_text = (
        "\n".join(next_month_lines[:10])
        if next_month_lines
        else "No data found for the next month – provide directional focus areas based on current trends."
    )

    return f"""
    You are an expert product manager and technical writer. Based on the following GitLab issues and
    new insights data, generate a comprehensive executive summary report.

    === GITLAB ISSUES DATA (all stories for this month) ===
    {issues_text}

    === TOP FEATURE DATA (issues labelled "Top Feature") ===
    {top_feature_text}

    === NEW INSIGHTS DATA (issues labelled "New Insight") ===
    {insights_text}

    === REQUIRED OUTPUT FORMAT ===
    Please analyse the data and provide a structured report. Use "<Question>: <answer>" style:

    1. Executive Summary:
    - What materially changed this month: <answer>
    - Why it matters to the business: <answer>

    2. Top Feature (or Insight) of the Month:
    - Insight name: <answer>

    3. New Insights (ranked by importance/impact):
    For each new insight:
    - Insight name: <answer>

    4. Process Improvements & Optimization:
    - Name / brief description: <answer>
    - How did we do it? (If AI was used, altered process, etc.): <answer>
    - Benefits: <answer>

    5. Platform Maintenance & Stability:
    - Maintenance, fixes, or technical improvements: <answer>
    - Why this matters (risk reduction, performance, cost control): <answer>

    6. Looking Ahead (Next Month):
    Use the following NEXT MONTH data (if available) to ground your answer:
    {next_month_text}
    - Focus areas (2-4 items max, focus on new insights coming next month): <answer>

    === INSTRUCTIONS ===
    - Be concise and business-focused
    - Rank insights by business impact and value
    - Use clear, non-technical language where possible
    - Focus on outcomes and benefits, not just features
    - If certain sections have no relevant data, state "No significant changes in this area"
    - Ensure all answers are data-driven based on the provided information
    - Section 1 Executive Summary: 2 sentences per question
    - Section 2 Top Feature: pick the single most impactful item from 'Top Feature' labelled issues
    - Section 3 New Insights: list each individual insight on a separate line, drawn from 'New Insight' labelled issues
    - Section 6 Looking Ahead: if no next-month data is available, provide directional focus areas based on current trends
    - Do NOT include instruction notes in the final output
    """.strip()


PROMPT = build_prompt(df_current, next_month_df=df_next_month)

# Preview the prompt (first 2 000 chars)
print(PROMPT[:2000])
print("..." if len(PROMPT) > 2000 else "")
print(f"\nTotal prompt length: {len(PROMPT)} characters")

In [ ]:
# ── Cell 8: Generate the executive summary ─────────────────────────────────

print(f"Calling {MODEL_NAME} … this may take up to 30 seconds.")

response = openai_client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": PROMPT}],
    max_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
)

SUMMARY_TEXT = response.choices[0].message.content.strip()

print("\n" + "=" * 80)
print(f"Executive Summary – {SELECTED_LABEL}")
print("=" * 80 + "\n")
print(SUMMARY_TEXT)

In [ ]:
# ── Cell 9: Save to .txt file (optional) ──────────────────────────────────

header = (
    f"Executive Summary Report – {SELECTED_LABEL}\n"
    f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
    + "=" * 80
    + "\n\n"
)

safe_month = SELECTED_MONTH.replace(" ", "_")
filename   = f"executive_summary_{safe_month}.txt"

with open(filename, "w", encoding="utf-8") as f:
    f.write(header + SUMMARY_TEXT)

print(f"Saved → {filename}")